# Support Vector Machine (SVM) Classification

## Definition
SVM is a **maximum-margin** supervised classification algorithm. It finds the **optimal hyperplane** that best separates classes by maximising the margin between the nearest data points of each class (called **support vectors**).

## Core Concept

### Linear SVM
The decision boundary: $\mathbf{w} \cdot \mathbf{x} + b = 0$

Objective: Maximise $\frac{2}{\|\mathbf{w}\|}$ (margin width)

### Soft Margin (C Parameter)
Allows some misclassifications via slack variables:
$$\text{Minimise } \frac{1}{2}\|w\|^2 + C \sum_i \xi_i$$

| C value | Effect |
|---------|--------|
| Large C | Narrow margin, fewer errors (risk of overfitting) |
| Small C | Wide margin, more errors tolerated |

### Kernel Trick
Maps data to higher-dimensional space to make non-linearly separable data separable.

| Kernel | Use Case |
|--------|----------|
| `linear` | Linearly separable data |
| `rbf` | Non-linear, general purpose |
| `poly` | Polynomial relationships |

## Applications
- **Bioinformatics** — Tumour classification, gene expression
- **Image recognition** — Face detection, handwriting recognition
- **Text categorisation** — Sentiment analysis, spam detection
- **Finance** — Stock market movement classification

---

## Why Breast Cancer Wisconsin Dataset?
> The dataset has **30 continuous features** and a **binary target** (malignant/benign). The feature space is high-dimensional, which is where SVM excels — it finds the maximum-margin hyperplane efficiently even in 30 dimensions. The dataset is also nearly linearly separable after scaling, making it a showcase for both `linear` and `rbf` kernels.

---

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import load_breast_cancer
from sklearn.svm import SVC
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, ConfusionMatrixDisplay

In [ ]:
data = load_breast_cancer()
df = pd.DataFrame(data.data, columns=data.feature_names)
df['target'] = data.target
print('Shape:', df.shape)
df.head()

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
X = df[data.feature_names]; y = df['target']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s  = scaler.transform(X_test)

# Compare kernels
for kernel in ['linear', 'rbf', 'poly']:
    svc = SVC(kernel=kernel, random_state=42)
    svc.fit(X_train_s, y_train)
    acc = accuracy_score(y_test, svc.predict(X_test_s))
    print(f'Kernel={kernel:<8}  Accuracy={acc:.4f}')

In [ ]:
# Best model: RBF kernel with tuned C & gamma
model = SVC(kernel='rbf', C=10, gamma='scale', probability=True, random_state=42)
model.fit(X_train_s, y_train)
y_pred = model.predict(X_test_s)

print('Accuracy:', accuracy_score(y_test, y_pred))
print()
print(classification_report(y_test, y_pred, target_names=data.target_names))

In [ ]:
cm = confusion_matrix(y_test, y_pred)
ConfusionMatrixDisplay(cm, display_labels=data.target_names).plot(cmap='Purples')
plt.title('SVM (RBF Kernel) — Confusion Matrix'); plt.tight_layout(); plt.show()

# Number of support vectors per class
print('Support vectors per class:', model.n_support_)
print('Total support vectors     :', model.n_support_.sum())